In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# 1. Charge les variables cachées dans le fichier .env
load_dotenv()

# 2. Récupère le jeton
mon_token = os.getenv("HF_TOKEN")

# 3. Authentifie la session courante
login(token=mon_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct"
 )
#template_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

In [3]:
from pathlib import Path

patterns = [
    # "data/1973/legislatives/*PF*.txt",
    # "data/1978/legislatives/*PF*.txt",
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt"
]

# 1. On charge d'abord tous vos fichiers dans un seul bloc global
dataset_complet = load_dataset("text", data_files=patterns, split="train", sample_by= "document")

# récupérer les fichiers dans le même ordre
files = []
for p in patterns:
    files.extend(sorted(Path().glob(p)))

def add_id(example, idx):
    path = files[idx]
    example["id"] = path.stem
    example["annee"] = path.parts[-3]
    example["election"] = path.parts[-2]
    return example

dataset_complet = dataset_complet.map(add_id, with_indices=True)



Resolving data files:   0%|          | 0/12498 [00:00<?, ?it/s]

In [4]:
import pandas as pd
metadata = pd.read_csv("data/archelect_search.csv")

In [5]:
metadata.columns

Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
       'cote', 'departement', 'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
      dtype='object')

In [6]:
metadata['titulaire-sexe'].value_counts(normalize=True)*100

titulaire-sexe
homme            83.725396
femme            10.449672
non déterminé     5.824932
Name: proportion, dtype: float64

In [7]:
metadata['contexte-tour'].value_counts(normalize=True)*100

contexte-tour
1    80.220835
2    19.779165
Name: proportion, dtype: float64

In [8]:
metadata['titulaire-profession'].value_counts(normalize=True)*100

titulaire-profession
non mentionné                       50.792127
professeur                           2.496399
avocat                               1.936310
chef d'entreprise                    1.520243
ingénieur                            1.288206
                                      ...    
chargé de mission départementale     0.008001
animatrice                           0.008001
coordonnateur                        0.008001
assistant direction                  0.008001
cadre juridique                      0.008001
Name: proportion, Length: 2032, dtype: float64

In [9]:
metadata['titulaire-soutien'].value_counts(normalize=True)*100

titulaire-soutien
non mentionné                                                                                                                                                     24.291887
Parti communiste français                                                                                                                                         12.257961
Front national                                                                                                                                                     9.865578
Parti socialiste                                                                                                                                                   7.913266
Rassemblement pour la République;Union pour la démocratie française                                                                                                6.072972
                                                                                                                          

In [10]:
# def add_soutien(example) : 
#     # print(metadata[metadata['id'] == example['id']]['titulaire-soutien'].values)
#     # print(example['id'])
#     metadata_dict = metadata.set_index('id').to_dict('index')
#     example['soutien'] = metadata[metadata['id'] == example['id']]['titulaire-soutien'].values[0]
#     example['prenom'] = metadata[metadata['id'] == example['id']]['titulaire-prenom'].values[0]
#     example['nom'] = metadata[metadata['id'] == example['id']]['titulaire-nom'].values[0]
#     example['profession'] = metadata[metadata['id'] == example['id']]['titulaire-profession'].values[0]
#     example['tour'] = metadata[metadata['id'] == example['id']]['contexte-tour'].values[0]
#     example['date'] = metadata[metadata['id'] == example['id']]['date'].values[0]
#     example['departement'] = metadata[metadata['id'] == example['id']]['departement-nom'].values[0]
    
#     return example

# dataset_complet = dataset_complet.map(add_soutien)

In [11]:
metadata_dict = metadata.set_index('id').to_dict('index')

In [12]:
def add_metadata(example) : 
    # print(metadata[example['id']]['titulaire-soutien'].values)
    # print(example['id'])
    example['soutien'] = metadata_dict[example['id']]['titulaire-soutien']
    example['prenom'] = metadata_dict[example['id']]['titulaire-prenom']
    example['nom'] = metadata_dict[example['id']]['titulaire-nom']
    example['profession'] = metadata_dict[example['id']]['titulaire-profession']
    example['tour'] = metadata_dict[example['id']]['contexte-tour']
    example['date'] = metadata_dict[example['id']]['date']
    example['departement'] = metadata_dict[example['id']]['departement-nom']
    return example

dataset_complet = dataset_complet.map(add_metadata)

In [13]:
# # 2. On divise ce bloc (ici : 10 % pour le test, 90 % pour l'entraînement)
# datasets_divises = dataset_complet.train_test_split(test_size=0.1, seed=42)

# # 3. On extrait nos deux sous-ensembles prêts à l'emploi !
# dataset_train = datasets_divises["train"]
# dataset_test = datasets_divises["test"]

In [14]:
metadata['titulaire-soutien'].value_counts()

titulaire-soutien
non mentionné                                                                                                                                                     3036
Parti communiste français                                                                                                                                         1532
Front national                                                                                                                                                    1233
Parti socialiste                                                                                                                                                   989
Rassemblement pour la République;Union pour la démocratie française                                                                                                759
                                                                                                                                                   

In [15]:
metadata['departement-nom'].isna().sum()

np.int64(0)

In [16]:
metadata[metadata['titulaire-nom'] == "non mentionné"]

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
2079,EL137_L_1981_06_088_01_1_PF_03,1981-06-14,Élections législatives;France;Ve République;As...,"Élections législatives de 1981, Vosges - 88, c...",législatives,1,EL137,88,Vosges,88 - Vosges,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Vosges écologie,non mentionné,non
2090,EL137_L_1981_06_088_04_1_PF_03,1981-06-14,France;Ve République;Élections législatives;As...,"Élections législatives de 1981, Vosges - 88, c...",législatives,1,EL137,88,Vosges,88 - Vosges,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Vosges écologie,non mentionné,non
2225,EL137_L_1981_06_092_06_1_PF_04,1981-06-14,France;Assemblée Nationale;Élections législati...,"Élections législatives de 1981, Hauts-de-Seine...",législatives,1,EL137,92,Hauts-de-Seine,92 - Hauts-de-Seine,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Aujourd'hui l'écologie,non
3284,EL174_L_1988_06_010_01_1_PF_06,1988-06-05,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1988, Aube - 10, cir...",législatives,1,EL174,10,Aube,10 - Aube,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Parti ouvrier européen,non mentionné,non
3289,EL174_L_1988_06_010_02_1_PF_05,1988-06-05,Assemblée Nationale;France;Élections législati...,"Élections législatives de 1988, Aube - 10, cir...",législatives,1,EL174,10,Aube,10 - Aube,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Parti ouvrier européen,non mentionné,non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11509,EL198_L_1993_03_095_07_1_PF_11,1993-03-21,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Nouveaux écologistes du rassemblement nature e...,non mentionné,non
11516,EL198_L_1993_03_095_08_1_PF_07,1993-03-21,Assemblée Nationale;Ve République;France;Élect...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non
11520,EL198_L_1993_03_095_08_1_PF_11,1993-03-21,Ve République;Assemblée Nationale;Élections lé...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Nouveaux écologistes du rassemblement nature e...,non mentionné,non
11526,EL198_L_1993_03_095_09_1_PF_06,1993-03-21,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non


In [17]:
#    example["prompt"] = (
#     f"Rédige une profession de foi pour {example['prenom']} {example['nom']}, "
#     f"de profession {example['profession']}, "
#     f"candidat soutenu par le parti {example['titulaire-soutien']} "
#     f"au tour {example['contexte-tour']} des élections législatives de {example['date']} "
#     f"dans le département : {example['departement-nom']}."
# )

In [18]:
metadata[metadata["titulaire-soutien"] == "Rassemblement pour la République;Union pour la démocratie française;Centre des démocrates sociaux;Centre national des indépendants"]

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
12442,EL198_L_1993_03_093_03_2_PF_01,1993-03-28,Ve République;Élections législatives;Assemblée...,"Élections législatives de 1993, Seine-Saint-De...",législatives,2,EL198,93,Seine-Saint-Denis,93 - Seine-Saint-Denis,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Rassemblement pour la République;Union pour la...,Opposition,non


In [19]:
dataset_complet.filter(lambda example : example['id'] == "EL198_L_1993_03_093_03_2_PF_01")[0]

{'text': 'Sciences Po / fonds CEVIPOF\nÉlections législatives d\'Aubervilliers - Le Bourget - La Courneuve\nFrédéric GAILLAND Non à l\'insécurité Oui à l\'emploi\nVous aussi, donnez un nouveau visage à la Seine S\' Denis\nEnsemble relançons la France Candidat soutenu par le CDS - UDF - RPR - CNISciences Po / fonds CEVIPOF\nPriorité à la sécurité et à l\'emploi pour la Seine ST Denis\nUn engagement personnel\nJe remercie toutes celles et tous ceux qui m\'ayant accordé leur confiance ou ayant voté pour des candidats démocrates, ont offert la chance d\'un véritable changement dans nos villes.\nDimanche prochain, c\'est la mobilisation de chacune et de chacun d\'entre nous qui mettra fin à plus d\'un demi siècle de "main mise communiste" sur notre circonscription.\nEnsemble, nous pourrons relancer la France pour créer des emplois, assurer la sécurité de tous, et redonner une vraie qualité de vie à chacun.\nCréer des emplois sera notre priorité en favorisant l\'investissement industriel sur

In [20]:
dataset_complet

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement'],
    num_rows: 12498
})

In [21]:
dataset_complet.filter(lambda example : example['nom'] == "Orabona")[0]

{'text': "Sciences Po / fonds CEVIPOF\n7 raisons de voter pour le candidat du Front National\nAntoine ORABONA Pharmacien - 32 ans\nPremière circonscription de la Haute-Vienne\nÉLECTIONS LÉGISLATIVES DU 5 JUIN 1988\nPour nous écrire : 8 rue du Général-Clergerie - 75116 PARIS\nSi vous voulez un député\n· qui soit présent à l'Assemblée nationale et qui remplisse effectivement le mandat pour lequel les contribuables le payent, comme les députés du Front National l'ont fait jusqu'ici à la différence de leurs collègues des autres groupes presque toujours absents ;\n· qui, une fois élu, ne s'entendra pas avec les socia- listes contre le vœu de ses électeurs, comme s'apprêtent déjà à le faire les centristes du RPR et de l'UDF :\n· qui se prononce clairement pour l'union de toutes les for- ces antisocialistes ;\n· qui vote pour la préférence nationale, la priorité d'emploi pour les Français, la suppression de la taxe professionnelle, la création du revenu maternel pour les mères de famille fran

In [22]:
def add_prompt(example) : 

    prompt = ["Rédige une profession de foi"]
    prenom = example['prenom'] 
    nom = example['nom']
    profession = example['profession']
    soutien = example['soutien']

    if prenom != "non mentionné" and nom != "non mentionné" : 
        prompt.append(f"pour {prenom} {nom},")
    else : 
        prompt.append("pour le candidat,")

    if profession != "non mentionné" : 
        liste_professions = profession.split(";")
        nb_professions = len(liste_professions)
        if nb_professions > 1 : 
            prompt.append("de professions")
            for i in range(nb_professions-1) :
                prompt.append(f"{liste_professions[i]},")
            prompt.append(f"et {liste_professions[-1]}")
        else : 
            prompt.append(f"de profession {profession},")
    
    if soutien != "non mentionné" : 
        liste_soutiens = soutien.split(';')
        nb_soutiens = len(liste_soutiens)
        if nb_soutiens > 1:
            prompt.append("soutenu par les partis")
            for i in range(nb_soutiens-1):
                prompt.append(f"{liste_soutiens[i]},")
            prompt.append(f"et {liste_soutiens[-1]}")
        else : 
            prompt.append(f"soutenu par le parti {soutien}")

    prompt.append(f"au tour {example['tour']} des élections législatives de {example['date']}")
    prompt.append(f"dans le département : {example['departement']}.")
    
    example['prompt'] = " ".join(prompt)
    
    return example

In [23]:
dataset_complet = dataset_complet.map(add_prompt)
dataset_complet

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement', 'prompt'],
    num_rows: 12498
})

In [24]:
import numpy as np
for i in np.random.randint(0, 12000, 10): 
    print(dataset_complet['prompt'][i])

Rédige une profession de foi pour Jacques Fuchs, de profession employé service public, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Rhône.
Rédige une profession de foi pour Robert Wagner, de profession ingénieur, soutenu par les partis Centre national des indépendants, Rassemblement pour la République, Centre des démocrates sociaux, et Parti radical au tour 1 des élections législatives de 1981-06-14 dans le département : Yvelines.
Rédige une profession de foi pour Jean-Pierre Kahane, de profession professeur mathématiques Université, soutenu par le parti Parti communiste français au tour 1 des élections législatives de 1988-06-05 dans le département : Essonne.
Rédige une profession de foi pour Hubert Guicharrousse, de profession universitaire, soutenu par les partis Verts, et Génération écologie au tour 1 des élections législatives de 1993-03-21 dans le département : Hauts-de-Seine.
Rédige une profession de foi po

In [25]:
# Data cleaning 

import re

def reparer_mots_coupes(texte):
    # RÈGLE 1 : Les tirets de fin de ligne (ex: "consolida-\ntion")
    # On cherche : (Des lettres) + un tiret + (des sauts de ligne ou espaces) + (Des lettres)
    # Le r'\1\2' dit à Python : recolle le groupe 1 (consolida) et le groupe 2 (tion) sans le tiret.
    # Note : Le [a-zA-ZÀ-ÿ] permet d'inclure les accents français (é, à, ç...)
    texte = re.sub(r'([a-zA-ZÀ-ÿ]+)-\s*\n\s*([a-zA-ZÀ-ÿ]+)', r'\1\2', texte)
    
    # RÈGLE 2 : Les tirets suivis d'un espace (ex: "plura- lisme")
    # On cherche : (Des lettres) + un tiret + (un ou plusieurs espaces) + (Des lettres)
    texte = re.sub(r'([a-zA-ZÀ-ÿ]+)-\s+([a-zA-ZÀ-ÿ]+)', r'\1\2', texte)
    
    return texte

# --- TEST POUR VÉRIFIER ---
texte_brut = """
maire-adjointe à Bourg-en-bresse
une étape importante pour la consolida-
tion de cette victoire.
le respect du plura- lisme politique.
- vive la France 
- et vive ce qu'il en reste
"""

print("AVANT :")
print(texte_brut)

print("APRÈS :")
print(reparer_mots_coupes(texte_brut))

AVANT :

maire-adjointe à Bourg-en-bresse
une étape importante pour la consolida-
tion de cette victoire.
le respect du plura- lisme politique.
- vive la France 
- et vive ce qu'il en reste

APRÈS :

maire-adjointe à Bourg-en-bresse
une étape importante pour la consolidation de cette victoire.
le respect du pluralisme politique.
- vive la France 
- et vive ce qu'il en reste



In [26]:
# import re

# pattern_cut_words = re.compile(r'([a-zA-ZÀ-ÿ]+)-\s*\n\s*([a-zA-ZÀ-ÿ]+)')
# pattern_new_line_words = re.compile(r'([a-zA-ZÀ-ÿ]+)-\s+([a-zA-ZÀ-ÿ]+)')
# pattern_side_words = re.compile(r'Sciences Po / fonds CEVIPOF|☐|☒|@|¥')
# pattern_vu_candidat = re.compile(r"vu(,?|\s+:\s+)\s+(le|la|les)\s+candidate?s?\s*[:.]?", flags=re.IGNORECASE)
# pattern_printing = re.compile(r".*imp(\.| ).*|.*(imprimerie|Imprimeurs|IMPR).*", flags=re.IGNORECASE)
# pattern_zipcode = re.compile(r"\b\d{5}\b")
# pattern_phone = re.compile(r"\b\d{1,2}([\s.-]?\d{2}){3}\b")
# pattern_company = re.compile(r".*(\s+r\.?c\.?\s+).*", flags=re.IGNORECASE)
# pattern_line_drop = re.compile(r'\n{3,}')
# pattern_multiple_spaces = re.compile(r' {2,}')
# #pattern_enumeration = re.compile(r"^\s*[•·.*>o]\s*", flags=re.MULTILINE)
# pattern_enumeration = re.compile(r"[:\n]\s*[•*>.o·]\s*", flags=re.MULTILINE)

# def clean_ocr(example):

#     text = example["text"]
#     # Les tirets de fin de ligne (ex: "consolida-\ntion")
#     text = pattern_cut_words.sub(r'\1\2', text)
#     #Les tirets suivis d'un espace (ex: "plura- lisme")
#     text = pattern_new_line_words.sub(r'\1\2', text)
#     # 2. Supprimer les filigranes CEVIPOF et les carrés magiques
#     text = pattern_side_words.sub("",text)
#     text = pattern_vu_candidat.sub("", text) 
#     # detects the printing company and drop these lines
#     text = pattern_printing.sub("", text)
#     # detect lines with phone numbers from years 1981 to 1993 and drop these lines
#     text = pattern_phone.sub("", text)
#     text = pattern_zipcode.sub("", text)
#     # detect lines with R.C. for companies and drop these lines
#     text = pattern_company.sub("", text)
#     # 3. Nettoyer les sauts de example excessifs (remplacer 3+ sauts de example par 2)
#     text = pattern_line_drop.sub('\n\n', text)
#     # 4. Enlever les espaces multiples
#     text = pattern_multiple_spaces.sub(' ', text)
#     # same format for all enumerations 
#     text = pattern_enumeration.sub("- ", text) # .
#     # enlever les codes postaux et les numéros de téléphone
#     # dégager les en-têtes
#     # On met à jour le text
#     example["text"] = text.strip()
    
#     return example

# # Application à tout le dataset
# dataset_propre = dataset_complet.map(clean_ocr)

In [27]:
import re

pattern_cut_words = re.compile(r'([A-Za-zÀ-ÿ]+)-\s*\n\s*([A-Za-zÀ-ÿ]+)')
pattern_new_line_words = re.compile(r'([A-Za-zÀ-ÿ]+)-\s+([A-Za-zÀ-ÿ]+)')

pattern_side_words = re.compile(r'Sciences Po / fonds CEVIPOF|☐|☒|@|¥|@')

pattern_vu_candidat = re.compile(
    r"vu\s*[,:\-]?\s*(le|la|les)\s+candidat[e]?[s]?\s*[:.]?",
    re.IGNORECASE
)

# lignes à supprimer complètement
pattern_drop_lines = re.compile(
    r"""
    ^.*(
        imp\.? |                      # imp / imp.
        imprimerie |
        imprimeurs |
        r\.?c\.? |                    # RC / R.C.
        \b\d{5}\b |                   # code postal
        \b\d{1,2}([\s.-]?\d{2}){3}\b  # téléphone
    ).*$
    """,
    re.IGNORECASE | re.MULTILINE | re.VERBOSE
)

pattern_line_drop = re.compile(r'\n{3,}')
pattern_multiple_spaces = re.compile(r' {2,}')

pattern_enumeration = re.compile(
    r"[:\n]\s*[•*>.o·]\s*",
    re.MULTILINE
)

def clean_ocr(example):

    text = example["text"]

    # réparer mots coupés OCR
    text = pattern_cut_words.sub(r'\1\2', text)
    text = pattern_new_line_words.sub(r'\1\2', text)

    # supprimer filigranes
    text = pattern_side_words.sub("", text)

    # supprimer mentions "Vu le candidat"
    text = pattern_vu_candidat.sub("", text)

    # supprimer lignes techniques (imprimeur, téléphone, RC, codes postaux)
    text = pattern_drop_lines.sub("", text)

    # nettoyer sauts de ligne
    text = pattern_line_drop.sub('\n\n', text)

    # supprimer espaces multiples
    text = pattern_multiple_spaces.sub(' ', text)

    # normaliser les énumérations
    text = pattern_enumeration.sub("- ", text)

    example["text"] = text.strip()

    return example


dataset_propre = dataset_complet.map(clean_ocr, num_proc=4)

In [28]:
import numpy as np
for i in np.random.randint(0, 12000, 1): 
    print(dataset_propre['text'][i])

LEGISLATIVES - 21 MARS 1993

RASSEMBLEMENT DES HOMMES ET DES FEMMES DE GAUCHE ET DE PROGRÈS
PARTI COMMUNISTE FRANÇAIS
SOLANGE
SCHMITTTRECANT
46 ans Ouvrière chez Bendix Militante syndicale Maire-adjointe de Beauvais
Suppléante
MARYSE BERTRAND
32 ans Employée chez Findus Militante syndicaleBeaucoup d'entre vous me le «ÇA NE PEUT

En 10 ans, notre région a perdu plus d'un emploi industriel sur cinq; près de 11% de la population active est au chômage.
Ce lamentable bilan est le fruit d'une politique qui a repris les vieilles recettes de la droite:
- austérité et chômage pour les uns
- cadeaux et privilèges pour le grand patronat et les financiers.

Leur donner votre voix, ce serait dire: "Allez-y ! continuez dans la même voie!".
D'ailleurs, j'ai le devoir de vous alerter sur le programme de la Droite RPR-UDF:
- démantèlement de l'Education Nationale et des services publics (hôpitaux, SNCF, Postes, EDF-GDF ... ]
 lisent : PLUS DURER»
- liberté totale de licenciement pour le patronat, et dé

In [29]:
dataset_complet.filter(lambda example : example['nom'] == "Poniatowski")[0]['text']

"ÉLECTIONS LÉGISLATIVES DU 14 JUIN 1981 2e CIRCONSCRIPTION DE L'EURE\nUn Nouveau Député Pour l'Eure\nLADISLAS PONIATOWSKI ET\nJEAN-JACQUES LEFORT\nCandidats de l'Union pour la Nouvelle Majorité. Candidats de l'Union pour la Démocratie Française.\nJean-Jacques LEFORT\nné le 5 septembre 1926 Marié - 3 enfants Agriculteur Conseiller Municipal de Bernay depuis 1965 Président Cantonal de la Fédération des Exploitants Agricoles Vice-Président de la Caisse du Crédit Agricole de Bernay\nLadislas PONIATOWSKI\nné le 10 novembre 1946 Marié - 2 enfants Licencié en Sciences Economiques Maîtrise de gestion - Cadre Maire de Quillebeuf-sur-Seine Président du Sivom Risle-Seine Vice-Président de l'association des Maires du Canton de Quillebeuf-sur-Seine\nPour un Député Présent et vraiment Actif à votre service.\nPour qu'une Majorité Libérale à l'Assemblée Nationale équilibre les pouvoirs d'un Président de la République socialiste.\nSciences Po / fonds CEVIPOFMES 18 PRIORITÉS\n1 - Maintenir les commerces

In [30]:
def add_messages(example) : 
    example['messages'] = [{"role" : "user", "content" : f"{example['prompt']}"},
                           {"role" : "assistant", "content" : f"{example['text']}"}]
    return example

In [31]:
dataset_propre = dataset_propre.map(add_messages)

In [32]:
dataset_propre['messages']

Column([[{'content': 'Rédige une profession de foi pour Micheline Antonucci, de profession assistance sociale, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Ain.', 'role': 'user'}, {'content': "Micheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\n\nmaire-adjointe à Bourg-en-bresse assistante sociale\n\nCela est nécessaire pour éviter des alliances centristes qui réduiraient à néant la victoire du 10 mai.\nDans cet esprit, le P.S.U. entend au premier tour de ces élections mettre l'accent sur des questions essentielles qui n'ont reçu, de la part de la gauche traditionnelle (P.S. et P.C.F.), que des réponses évasives ou des refus. Au deuxième tour, fidèle à ses engagements, il sera aux côtés du candidat de la gauche le mieux placé pour assurer la victoire contre la droit

In [33]:
def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""
    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

In [34]:
dataset_propre = dataset_propre.map(format_prompt)

In [35]:
dataset_propre['text'][0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Apr 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nRédige une profession de foi pour Micheline Antonucci, de profession assistance sociale, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Ain.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\n\nmaire-adjointe à Bourg-en-bresse assistante sociale\n\nCela est nécessaire pour éviter des alliances centristes qui réduiraient à néant la victoire du 10 mai.\nDans cet esprit, le P.S.U. entend au premier tour de ces élections mettre l'accent sur des questions essentielles qui n'ont reçu, de la part de la gauche traditionnelle (P.S. et P.C

In [36]:
dataset_propre

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement', 'prompt', 'messages'],
    num_rows: 12498
})

In [38]:
dataset_test

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement', 'prompt', 'messages'],
    num_rows: 100
})

In [39]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# # 4-bit quantization configuration - Q in QLoRA
# bnb_config = BitsAndBytesConfig(
# load_in_4bit=True, # Use 4-bit precision model loading
# bnb_4bit_quant_type="nf4", # Quantization type
# bnb_4bit_compute_dtype="float16", # Compute dtype
# bnb_4bit_use_double_quant=True, # Apply nested quantization
# )
# # Load the model to train on the GPU
# model = AutoModelForCausalLM.from_pretrained(
# model_name,
# device_map="auto",
# # Leave this out for regular SFT
# quantization_config=bnb_config,
# )
# model.config.use_cache = False
# model.config.pretraining_tp = 1
# # Load LLaMA tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = "<PAD>"
# tokenizer.padding_side = "left"

In [40]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "meta-llama/Llama-3.2-1B-Instruct"

# # 1. Chargement et configuration du Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# # On recycle le jeton de fin existant au lieu d'en inventer un
# tokenizer.pad_token = tokenizer.eos_token 
# tokenizer.padding_side = "right" # Pour l'entraînement (Causal LM), on ajoute le remplissage à la fin

# # 2. Chargement du Modèle (optimisé pour Apple Silicon)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",           # On envoie directement sur la puce graphique de votre Mac
#     torch_dtype=torch.bfloat16  # Précision 16-bit : le modèle pèsera environ 2.5 Go en RAM
# )
# model.config.use_cache = False
# model.config.pretraining_tp = 1

In [41]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 1. Configuration QLoRA (4-bit) indispensable pour la Tesla T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # float16 au lieu de bfloat16
    bnb_4bit_use_double_quant=True,
)

# 2. Chargement du Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "right" # RIGHT pour l'entraînement (SFT) !

# 3. Chargement du Modèle en 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config, # On injecte la config 4-bit
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# 4. Préparation spécifique pour l'entraînement en 4-bit
model = prepare_model_for_kbit_training(model)

/opt/python/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/opt/python/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/opt/python/lib/python3.13/site-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [42]:
from peft import LoraConfig, get_peft_model
# Prepare LoRA Configuration
peft_config = LoraConfig(
lora_alpha=32, # LoRA Scaling
lora_dropout=0.1, # Dropout for LoRA Layers
r=64, # Rank
bias="none",
task_type="CAUSAL_LM",
target_modules= # Layers to target
["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
"down_proj"]
)
# Prepare model for training
#model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
# from transformers import TrainingArguments
# output_dir = "./results"
# # Training arguments
# training_arguments = TrainingArguments(
# output_dir=output_dir,
# per_device_train_batch_size=2,
# gradient_accumulation_steps=4,
# optim="paged_adamw_32bit",
# #optim = "adamw_torch",
# learning_rate=2e-4,
# lr_scheduler_type="cosine",
# num_train_epochs=1,
# logging_steps=10,
# #fp16=True,
# bf16=True,
# gradient_checkpointing=True
# )

In [43]:
dataset_light = dataset_propre.remove_columns(['prompt'])

In [44]:
dataset_light

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement', 'messages'],
    num_rows: 12498
})

In [45]:
dataset_test = dataset_light.select(range(100))

In [ ]:
# from trl import SFTTrainer, SFTConfig
# # Set supervised fine-tuning parameters
# training_arguments = SFTConfig(
# output_dir=output_dir,
# per_device_train_batch_size=2,
# gradient_accumulation_steps=4,
# optim="paged_adamw_32bit",
# #optim = "adamw_torch",
# learning_rate=2e-4,
# lr_scheduler_type="cosine",
# num_train_epochs=1,
# logging_steps=10,
# #fp16=True,
# bf16=True,
# gradient_checkpointing=True,
# dataset_text_field="text",
# max_length=512
# )
# trainer = SFTTrainer(
# model=model,
# train_dataset=dataset_light,
# processing_class=tokenizer,
# args=training_arguments,
# # Leave this out for regular SFT
# #peft_config=peft_config,
# )
# # Train model
# trainer.train()
# # Save QLoRA weights
# trainer.model.save_pretrained("Llama-1B-Archelec-LoRA")

Tokenizing train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


KeyboardInterrupt: 

In [47]:
from trl import SFTTrainer, SFTConfig

output_dir = "./results"

training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,           # fp16 au lieu de bf16 pour la Tesla T4 !
    bf16=False,          # On s'assure que bf16 est bien désactivé
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=2048, # 2048 au lieu de 512 pour ne pas couper les textes. (SFTConfig utilise max_seq_length)
    packing=False        # Bonne pratique explicite pour ce type de dataset
)

# L'initialisation du Trainer reste identique
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_light,
    processing_class=tokenizer, # Attention, 'processing_class' remplace 'tokenizer' dans les nouvelles versions de TRL
    args=training_arguments,
    peft_config=peft_config, # N'oublie pas de décommenter ça !
)

# Go !
trainer.train()

# Sauvegarde
trainer.model.save_pretrained("Llama-1B-Archelec-LoRA")
tokenizer.save_pretrained("Llama-1B-Archelec-LoRA") # Sauvegarde aussi le tokenizer par sécurité

ValueError: You passed a `PeftModel` instance together with a `peft_config` to the trainer. Please first merge and unload the existing adapter, save the resulting base model, and then pass that base model along with the new `peft_config` to the trainer.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Chemins
base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "Llama-1B-Archelec-LoRA"

# 2. Configuration pour la Tesla T4 (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. Chargement du Tokenizer
# Très important : pour l'inférence (génération), le padding doit être à GAUCHE ("left")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.padding_side = "left"

# 4. Chargement du modèle de base
print("Chargement du modèle de base...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

# 5. Fusion (virtuelle) avec tes poids LoRA
print("Application des poids fine-tunés (LoRA)...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 6. Préparation du Prompt de test
test_prompt = "Rédige une profession de foi pour Jean Dupont, de profession professeur, soutenu par le parti Parti communiste français au tour 1 des élections législatives de 1981-06-14 dans le département : Paris."

messages = [
    {"role": "user", "content": test_prompt}
]

# add_generation_prompt=True ajoute automatiquement le tag <|assistant|> à la fin pour dire au modèle de parler
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

print("\n" + "="*50)
print("GÉNÉRATION : MODÈLE FINE-TUNÉ (Ton Archelec)")
print("="*50)
# Génération avec le modèle entraîné
outputs_ft = model.generate(
    **inputs, 
    max_new_tokens=400,   # Longueur de la réponse générée
    temperature=0.7,      # Un peu de créativité (0.0 = très strict, 1.0 = très créatif)
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)
response_ft = tokenizer.decode(outputs_ft[0], skip_special_tokens=True)
# On isole la réponse de l'assistant (pour ne pas réafficher le prompt)
print(response_ft.split("assistant\n")[-1] if "assistant\n" in response_ft else response_ft)


print("\n" + "="*50)
print("GÉNÉRATION : MODÈLE DE BASE (Llama-3.2-1B-Instruct classique)")
print("="*50)
# Génération sans ton entraînement (on désactive temporairement le LoRA)
with model.disable_adapter():
    outputs_base = model.generate(
        **inputs, 
        max_new_tokens=400, 
        temperature=0.7, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
response_base = tokenizer.decode(outputs_base[0], skip_special_tokens=True)
print(response_base.split("assistant\n")[-1] if "assistant\n" in response_base else response_base)